# Skyline Online Courses: Hypothesis Tests

Lesson 1.6 Practice Exercise. Runs t-tests on the Skyline Online Courses dataset,
demonstrates the multiple testing problem with simulation, and shows how
Bonferroni correction works.

Author: Kenya Harvey
Date: 08-13-2026

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [9]:
skyline = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")

Python_for_Beginners = skyline[skyline["course_name"] == "Python for Beginners"]["hours_studied"]
SQL_basics = skyline[skyline["course_name"] == "SQL Basics"]["hours_studied"]

print(f"Python for Beginners: n={len(Python_for_Beginners)}, mean={Python_for_Beginners.mean():.2f}, std={Python_for_Beginners.std():.2f}")
print(f"SQL Basics: n={len(SQL_basics)}, mean={SQL_basics.mean():.2f}, std={SQL_basics.std():.2f}")

Python for Beginners: n=20, mean=30.71, std=3.30
SQL Basics: n=18, mean=28.91, std=2.04


In [4]:
t_stat, p_value = stats.ttest_ind(Python_for_Beginners, SQL_basics)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 2.0016
p-value: 0.0529


In [10]:
mean_diff = Python_for_Beginners.mean() - SQL_basics.mean()

# Pooled standard error of the difference
n1, n2 = len(Python_for_Beginners), len(SQL_basics)
s1, s2 = Python_for_Beginners.std(ddof=1), SQL_basics.std(ddof=1)
se_diff = np.sqrt(s1**2 / n1 + s2**2 / n2)

# 95% CI for the difference
margin = 1.96 * se_diff
ci_lower = mean_diff - margin
ci_upper = mean_diff + margin

print(f"Mean difference (Python for Beginners - SQL Basics): {mean_diff:.2f}")
print(f"95% CI for the difference: ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"p-value: {p_value:.4f}")

Mean difference (Python for Beginners - SQL Basics): 1.80
95% CI for the difference: (0.08, 3.53)
p-value: 0.0529


### Interpretation

The average difference in hours studied between Python for Beginners and SQL Basics was 1.80 hours. Python for Beginners had more hours studied. Because the p-value 0.0529 is greater than the 0.05 significance threshold, this difference was not statistically detectable at the 5% level. Because the mean difference of 1.80 hours exceeds half of both standard deviations (1.65 and 1.02), this represents a medium to large effect size. Given the small sample sizes (n=20, n=18) and a p-value just above 0.05, this borderline result doesn't rule out a real difference, it just means the study didn't have enough data to confirm it with full confidence.

In [1]:
import itertools

courses = ["Python for Beginners", "SQL Basics", "Intro to Analytics", "Statistics 101", "Tableau Fundamentals"]
pairs = list(itertools.combinations(courses, 2))
print(pairs)
print(len(pairs))

[('Python for Beginners', 'SQL Basics'), ('Python for Beginners', 'Intro to Analytics'), ('Python for Beginners', 'Statistics 101'), ('Python for Beginners', 'Tableau Fundamentals'), ('SQL Basics', 'Intro to Analytics'), ('SQL Basics', 'Statistics 101'), ('SQL Basics', 'Tableau Fundamentals'), ('Intro to Analytics', 'Statistics 101'), ('Intro to Analytics', 'Tableau Fundamentals'), ('Statistics 101', 'Tableau Fundamentals')]
10


In [8]:
from scipy import stats

if "skyline" not in globals():
    skyline = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")

results = []
all_p_values = []

for course_a, course_b in pairs:
    hours_a = skyline[skyline["course_name"] == course_a]["hours_studied"]
    hours_b = skyline[skyline["course_name"] == course_b]["hours_studied"]

    mean_diff = hours_a.mean() - hours_b.mean()
    t_stat, p_value = stats.ttest_ind(hours_a, hours_b)

    results.append({
        "course_a": course_a,
        "course_b": course_b,
        "mean_difference": mean_diff,
        "p_value": p_value
    })
    all_p_values.append(p_value)

results_df = pd.DataFrame(results)
n_tests = len(all_p_values)
significant_count = sum(p < 0.05 for p in all_p_values)

print(results_df)

               course_a              course_b  mean_difference   p_value
0  Python for Beginners            SQL Basics         1.804444  0.052905
1  Python for Beginners    Intro to Analytics         1.639167  0.035871
2  Python for Beginners        Statistics 101         0.063333  0.957179
3  Python for Beginners  Tableau Fundamentals         2.723636  0.005497
4            SQL Basics    Intro to Analytics        -0.165278  0.766385
5            SQL Basics        Statistics 101        -1.741111  0.090147
6            SQL Basics  Tableau Fundamentals         0.919192  0.241424
7    Intro to Analytics        Statistics 101        -1.575833  0.065843
8    Intro to Analytics  Tableau Fundamentals         1.084470  0.099065
9        Statistics 101  Tableau Fundamentals         2.660303  0.014639


In [9]:
bonferroni_threshold = 0.05 / n_tests
significant_after_bonferroni = sum(p < bonferroni_threshold for p in all_p_values)

print(f"Original threshold: 0.05")
print(f"Bonferroni-corrected threshold: {bonferroni_threshold:.4f}")
print(f"Number significant at original threshold: {significant_count}")
print(f"Number significant after Bonferroni: {significant_after_bonferroni}")

Original threshold: 0.05
Bonferroni-corrected threshold: 0.0050
Number significant at original threshold: 3
Number significant after Bonferroni: 0


## Part B: Multiple Testing in Practice

Three out of 10 tests looked significant, but the courses are actually similar (drawn from comparable distributions), meaning any "true" differences are essentially zero. Running multiple tests increases the odds that at least one will cross the 0.05 threshold purely by chance, even when nothing real is happening. This teaches me that when running multiple comparisons, I should report all the tests I ran, not just the ones that came back significant and apply a correction like Bonferroni before treating any single result as trustworthy.